## Unzipping

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Unzip the bundle to Colab's high-speed local SSD
!unzip -q /content/drive/MyDrive/Namrata-IMP-Docs/IIITH/LMA_Project_Phase1_data/Tokenizers/tokenizers_colab_bundle.zip -d /content/drive/MyDrive/Namrata-IMP-Docs/IIITH/LMA_Project_Phase1_data/Tokenizers/

In [1]:
from google.colab import files

uploaded = files.upload()

Saving Tokenizers-rq1.zip to Tokenizers-rq1.zip


In [2]:
import zipfile
import os

zip_path = "/content/Tokenizers-rq1.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir("/content"))

['.config', 'Tokenizers-rq1', 'Tokenizers-rq1.zip', 'sample_data']


## Running baseline (test_only - 120 samples)

In [5]:
# Move into the pipeline directory
%cd /content/Tokenizers-rq1/Tokenizers
!ls -la

/content/Tokenizers-rq1/Tokenizers
total 128
drwxr-xr-x 4 root root   4096 Oct 28 11:06  .
drwxr-xr-x 3 root root   4096 Oct 28 11:06  ..
drwxr-xr-x 4 root root   4096 Oct 28 11:06  aci_dataset_rq1_baseline
-rw-r--r-- 1 root root 111020 Oct 28 11:06 "tokenizer's_umedsum_baseline.ipynb"
drwxr-xr-x 6 root root   4096 Oct 28 11:06  umedsum_pipeline_baseline


In [7]:
# Change directory directly to your Google Drive pipeline folder
%cd /content/Tokenizers-rq1/Tokenizers/umedsum_pipeline_baseline

# Verify files are visible
!ls -la


/content/Tokenizers-rq1/Tokenizers/umedsum_pipeline_baseline
total 100
drwxr-xr-x 6 root root  4096 Oct 28 11:06 .
drwxr-xr-x 4 root root  4096 Oct 28 11:06 ..
-rw-r--r-- 1 root root 21653 Oct 28 11:06 evaluate_metrics.py
-rw-r--r-- 1 root root 20884 Oct 28 11:06 extracted_paper_plan.txt
-rw-r--r-- 1 root root 12485 Oct 28 11:06 README.md
-rw-r--r-- 1 root root   345 Oct 28 11:06 requirements.txt
drwxr-xr-x 2 root root  4096 Oct 28 11:06 results
-rw-r--r-- 1 root root  5468 Oct 28 11:06 run_all_stages.py
drwxr-xr-x 3 root root  4096 Oct 28 11:06 stage_1
drwxr-xr-x 3 root root  4096 Oct 28 11:06 stage_2
drwxr-xr-x 3 root root  4096 Oct 28 11:06 stage_3


In [8]:
# 1. Install zstd (required by Ollama) and then install Ollama
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Install required Python packages
!pip install -q torch transformers sentence-transformers keybert pysbd bert-score rouge-score accelerate

# 3. Start Ollama daemon in background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(4)

# 4. Pull LLaMA 3 8B onto Colab's GPU
!ollama pull llama3:8b


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/loca

In [9]:
# ===== Extra setup for SummaC / QuestEval / Entailment metrics =====
import os, sys, subprocess, importlib

# 1. Supporting libraries
!pip install -q evaluate unidecode sentencepiece nltk datasets spacy
!python -m spacy download en_core_web_sm -q
import nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)

# 2. SummaC (without its old pinned dependencies)
!pip install -q summac --no-deps

# 3. QuestEval: do NOT pip install it. Clone the repo, patch it, and use it from the path
subprocess.run("pip uninstall -y -q questeval", shell=True)      # remove any stale empty install
subprocess.run("rm -rf /content/QuestEval", shell=True)
subprocess.run("git clone --depth 1 https://github.com/ThomasScialom/QuestEval.git /content/QuestEval", shell=True, check=True)
subprocess.run(
    "sed -i 's/from datasets import load_metric/from evaluate import load as load_metric/' "
    "/content/QuestEval/questeval/questeval_metric.py", shell=True, check=True)

# 4. Make Python (and later `!python evaluate_metrics.py`) find the clone
os.environ["PYTHONPATH"] = "/content/QuestEval:" + os.environ.get("PYTHONPATH", "")
for k in [k for k in sys.modules if k.startswith("questeval")]:
    del sys.modules[k]                                           # drop any cached stale package
sys.path.insert(0, "/content/QuestEval")
importlib.invalidate_caches()

# 5. Verify
import torch
print("GPU available:", torch.cuda.is_available())
for m in ["summac.model_summac", "questeval.questeval_metric"]:
    try:
        mod = importlib.import_module(m)
        print(m, "OK", "->", getattr(mod, "__file__", ""))
    except Exception as e:
        print(m, "FAILED ->", type(e).__name__, e)


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
GPU available: True
summac.model_summac OK -> /usr/local/lib/python3.13/dist-packages/summac/model_summac.py
questeval.questeval_metric OK -> /content/QuestEval/questeval/questeval_metric.py


/content/QuestEval/questeval/utils.py:359: SyntaxWarning: invalid escape sequence '\s'
  s = re.sub('\s+', '_', s)  # turn spaces to underscores
/content/QuestEval/questeval/utils.py:360: SyntaxWarning: invalid escape sequence '\s'
  s = re.sub('\s+\(in metres\)', '_m', s)
/content/QuestEval/questeval/utils.py:361: SyntaxWarning: invalid escape sequence '\s'
  s = re.sub('\s+\(in feet\)', '_f', s)
/content/QuestEval/questeval/utils.py:362: SyntaxWarning: invalid escape sequence '\('
  s = re.sub('\(.*\)', '', s)


In [10]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("Active GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA Available: True
Active GPU: Tesla T4


In [11]:
# !python run_all_stages.py --limit 1
!python run_all_stages.py --input-json ../aci_dataset_rq1_baseline/test_only/test.json


  EXECUTING STAGE 1 (INITIAL GENERATION - PAPER SOTA)
Command: /usr/bin/python3 /content/Tokenizers-rq1/Tokenizers/umedsum_pipeline_baseline/stage_1/stage1_generator.py --input-json ../aci_dataset_rq1_baseline/test_only/test.json --output-json /content/Tokenizers-rq1/Tokenizers/umedsum_pipeline_baseline/stage_1/stage1_output.json --n-shots 1
Connecting to Ollama at http://localhost:11434...
Connected! Available models in Ollama: ['llama3:8b']
Loaded 1-shot ICL demonstration from icl_examples.json
Resuming: found 140 previously completed samples in stage1_output.json

Running Paper SOTA (Element-Aware + ICL) on 120 samples...
  Model  : llama3:8b
  Input  : ../aci_dataset_rq1_baseline/test_only/test.json
  Output : /content/Tokenizers-rq1/Tokenizers/umedsum_pipeline_baseline/stage_1/stage1_output.json


Stage 1 complete! Successfully generated 140 summaries in 0.0s.
Results saved to: /content/Tokenizers-rq1/Tokenizers/umedsum_pipeline_baseline/stage_1/stage1_output.json

  EXECUTING ST

In [ ]:
!python run_all_stages.py --input-json ../aci_dataset_rq1_baseline/test+val/test_val_shuffled.json

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
!cp -r /content/Tokenizers-rq1 "/content/drive/MyDrive/Tokenizers-rq1_backup"
!ls "/content/drive/MyDrive/Tokenizers-rq1_backup/Tokenizers/umedsum_pipeline_baseline/results"

evaluation_results.json  stage2_output.json	   stage3_output.json
stage1_output.json	 stage3_final_scores.json
